# Embedding and Index Design
Companion notebook for the **Embedding and Index Design** section of Chapter 10.

Chunking decides *what* can be found; embeddings decide *how* it gets matched. This notebook needs no API key or ML library: it uses a small, deterministic **hashed bag-of-words** embedding (the same "hashing trick" real systems like scikit-learn's `HashingVectorizer` use) -- similar wording lands closer together, exactly like a real embedding model, just without needing one installed.

In [1]:
import hashlib
import math

EMBED_DIM = 32


def embed_text(text: str, dim: int = EMBED_DIM) -> list[float]:
    """Deterministic stand-in for a real embedding model: each word hashes into one
    dimension, so texts sharing more words end up with a higher cosine similarity."""
    vector = [0.0] * dim
    for word in text.lower().split():
        idx = int(hashlib.sha256(word.encode()).hexdigest(), 16) % dim
        vector[idx] += 1.0
    norm = math.sqrt(sum(v * v for v in vector)) or 1.0
    return [v / norm for v in vector]


def cosine_similarity(a: list[float], b: list[float]) -> float:
    return sum(x * y for x, y in zip(a, b))


v1 = embed_text("Refunds are allowed within 30 days of purchase")
v2 = embed_text("How do I get my money back after a purchase")
v3 = embed_text("The weather today is sunny with a light breeze")
print("refund vs refund-paraphrase similarity:", round(cosine_similarity(v1, v2), 3))
print("refund vs unrelated similarity:         ", round(cosine_similarity(v1, v3), 3))

refund vs refund-paraphrase similarity: 0.144
refund vs unrelated similarity:          0.075


The book's practical record -- a chunk carries its embedding **and** operational metadata, not just the vector:

In [2]:
chunk_text = "Refunds are allowed within 30 days of purchase for annual plans."

record = {
    "chunk_id": "doc-42#chunk-7",
    "doc_id": "doc-42",
    "text": chunk_text,
    "embedding": embed_text(chunk_text),
    "section": "Returns Policy",
    "version": "2026-04-01",
    "tenant_id": "acme",
    "acl": ["support", "manager"],
}
print({**record, "embedding": f"<{len(record['embedding'])}-dim vector>"})

{'chunk_id': 'doc-42#chunk-7', 'doc_id': 'doc-42', 'text': 'Refunds are allowed within 30 days of purchase for annual plans.', 'embedding': '<32-dim vector>', 'section': 'Returns Policy', 'version': '2026-04-01', 'tenant_id': 'acme', 'acl': ['support', 'manager']}


A tiny in-memory vector store -- `upsert` and metadata-filtered `search` -- standing in for a real vector database:

In [3]:
class InMemoryVectorStore:
    def __init__(self):
        self._records: dict[str, dict] = {}

    def upsert(self, record: dict) -> None:
        self._records[record["chunk_id"]] = record

    def search(self, query_vector: list[float], top_k: int = 5, filters: dict | None = None) -> list[dict]:
        candidates = self._records.values()
        if filters:
            candidates = [r for r in candidates if all(r.get(k) == v for k, v in filters.items())]
        scored = [{**r, "score": cosine_similarity(query_vector, r["embedding"])} for r in candidates]
        scored.sort(key=lambda r: r["score"], reverse=True)
        return scored[:top_k]


vector_store = InMemoryVectorStore()
vector_store.upsert(record)
vector_store.upsert({
    "chunk_id": "doc-43#chunk-1",
    "doc_id": "doc-43",
    "text": "Enterprise plans follow a separate 45 day refund window.",
    "embedding": embed_text("Enterprise plans follow a separate 45 day refund window."),
    "section": "Enterprise Terms",
    "version": "2026-04-01",
    "tenant_id": "globex",
    "acl": ["support"],
})

hits = vector_store.search(embed_text("what is the refund window"), top_k=5, filters={"tenant_id": "acme"})
for h in hits:
    print(f"{h['chunk_id']}  score={h['score']:.3f}  tenant={h['tenant_id']}  text={h['text'][:50]!r}")

doc-42#chunk-7  score=0.217  tenant=acme  text='Refunds are allowed within 30 days of purchase for'


Filtering by `tenant_id="acme"` correctly excludes the Globex chunk even though it's topically relevant -- exactly the tenant-isolation the book insists on. A **flat** index like this is simple and exact but slow at scale; approximate indexes like **HNSW** or **IVF** trade a little recall for much lower latency on large collections -- worth knowing about, but out of scope for this small demo.

Version your embeddings and indexes rather than mutating one collection in place -- build the new version in parallel, validate it, then switch traffic:

In [4]:
def build_versioned_record(chunk_id: str, chunk_text: str, metadata: dict) -> dict:
    return {
        "chunk_id": chunk_id,
        "text": chunk_text,
        "embedding_model": "hashed-bow-v1",
        "embedding_version": "2026-04",
        "metadata": metadata,
    }


versioned_record = build_versioned_record(
    "doc-42#chunk-7", chunk_text, {"tenant_id": "acme", "section": "Returns Policy"}
)
print(versioned_record)

{'chunk_id': 'doc-42#chunk-7', 'text': 'Refunds are allowed within 30 days of purchase for annual plans.', 'embedding_model': 'hashed-bow-v1', 'embedding_version': '2026-04', 'metadata': {'tenant_id': 'acme', 'section': 'Returns Policy'}}
